# Generate a ManiSkill PickCube LeRobot Dataset

This notebook generates a reproducible **pipeline/smoke-test** dataset directly from `PickCube-v1`, writes a raw HDF5 copy, converts it to LeRobot v3 format, and validates the result. Random actions are intentionally labelled as non-expert data: use them to test schemas and loaders, not to train a useful imitation-learning policy.

In [1]:
from pathlib import Path

# Notebooks may be started with either the repository root or notebooks/ as the working
# directory. Walk up until the project root is found, so dataset and cache paths never
# resolve to notebooks/datasets or notebooks/.cache by accident.
_cwd = Path.cwd().resolve()
PROJECT_ROOT = next(
    (p for p in (_cwd, *_cwd.parents) if (p / ".git").exists()),
    _cwd.parent if _cwd.name == "notebooks" else _cwd,
)
import shutil
import numpy as np
import h5py
import torch
import gymnasium as gym
import mani_skill.envs  # registers ManiSkill environments
from mani_skill.utils.wrappers.gymnasium import CPUGymWrapper
# Set the Hugging Face cache BEFORE importing lerobot: the `datasets` package snapshots
# its cache location at import time, so setting the environment later has no effect and
# reads fall back to the (non-writable) home directory.
import os

HF_CACHE = PROJECT_ROOT / ".cache" / "hf"
(HF_CACHE / "datasets").mkdir(parents=True, exist_ok=True)
os.environ.setdefault("HF_HOME", str(HF_CACHE))
os.environ.setdefault("HF_DATASETS_CACHE", str(HF_CACHE / "datasets"))

from lerobot.datasets.lerobot_dataset import LeRobotDataset

ENV_ID = 'PickCube-v1'
OBS_MODE = 'state'
CONTROL_MODE = 'pd_joint_delta_pos'
NUM_EPISODES = 10
MAX_STEPS = 50
FPS = 50
SEED = 42
TASK = 'Pick up the cube and move it to the goal position.'
REPO_ID = 'local/maniskill_pickcube_smoke'
OUTPUT_ROOT = PROJECT_ROOT / 'datasets' / 'lerobot' / 'maniskill_pickcube_smoke'
RAW_H5 = PROJECT_ROOT / 'datasets' / 'raw' / 'pickcube_smoke.h5'
OVERWRITE = True
print('Output:', OUTPUT_ROOT)

Output: /home/bowenyuan/Projects/embodied-ai-learning/datasets/lerobot/maniskill_pickcube_smoke


In [2]:
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
print('Dataset mode: RANDOM SMOKE TEST — not expert demonstrations')

PyTorch: 2.11.0+cu128
CUDA available: False
Dataset mode: RANDOM SMOKE TEST — not expert demonstrations


/home/bowenyuan/miniforge3/envs/embodied/lib/python3.12/site-packages/torch/cuda/__init__.py:180: UserWarning: CUDA initialization: Unexpected error from cudaGetDeviceCount(). Did you run some cuda functions before calling NumCudaDevices() that might have already set an error? Error 304: OS call failed or operation not supported on this OS (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0


In [3]:
def as_numpy(x):
    if isinstance(x, torch.Tensor):
        x = x.detach().cpu().numpy()
    return np.asarray(x)

def scalar(x):
    return float(as_numpy(x).reshape(-1)[0])

def boolean(x):
    return bool(as_numpy(x).reshape(-1)[0])

def collect_episode(env, seed, max_steps):
    obs, info = env.reset(seed=seed)
    observations, actions, rewards, timestamps = [], [], [], []
    terminated_flags, truncated_flags, success_flags = [], [], []
    rng = np.random.default_rng(seed)
    low = np.asarray(env.action_space.low, dtype=np.float32)
    high = np.asarray(env.action_space.high, dtype=np.float32)
    for step in range(max_steps):
        action = rng.uniform(low, high).astype(np.float32)
        observations.append(as_numpy(obs).astype(np.float32).reshape(-1))
        actions.append(action.reshape(-1))
        timestamps.append(step / FPS)
        obs, reward, terminated, truncated, info = env.step(action)
        rewards.append(scalar(reward))
        terminated_flags.append(boolean(terminated))
        truncated_flags.append(boolean(truncated))
        success_flags.append(boolean(info.get('success', False)))
        if terminated_flags[-1] or truncated_flags[-1]:
            break
    return {
        'observations': np.stack(observations),
        'actions': np.stack(actions),
        'rewards': np.asarray(rewards, dtype=np.float32),
        'timestamps': np.asarray(timestamps, dtype=np.float64),
        'terminated': np.asarray(terminated_flags, dtype=bool),
        'truncated': np.asarray(truncated_flags, dtype=bool),
        'success': np.asarray(success_flags, dtype=bool),
    }

In [4]:
env = gym.make(
    ENV_ID, num_envs=1, obs_mode=OBS_MODE,
    control_mode=CONTROL_MODE, render_mode=None
)
env = CPUGymWrapper(env)
episodes = [collect_episode(env, SEED + i, MAX_STEPS) for i in range(NUM_EPISODES)]
env.close()
obs_dim = episodes[0]['observations'].shape[1]
action_dim = episodes[0]['actions'].shape[1]
print(f'Collected {len(episodes)} episodes; observation={obs_dim}D, action={action_dim}D')
print('Frames:', sum(len(ep['actions']) for ep in episodes))
print('Successful episodes:', sum(bool(ep['success'].any()) for ep in episodes))

2026-09-22 11:36:17,290 - mani_skill  - WARNING - Requested to use render device "sapien_cuda", but CUDA device was not found. Falling back to "cpu" device. Rendering might be disabled.


/home/bowenyuan/miniforge3/envs/embodied/lib/python3.12/site-packages/torch/cuda/__init__.py:1061: UserWarning: Can't initialize NVML
  raw_cnt = _raw_device_count_nvml()


Collected 10 episodes; observation=42D, action=8D
Frames: 500
Successful episodes: 0


### Raw HDF5 intermediate

The episodes above are first stored as raw HDF5: the format the simulator
produced, before any training-oriented schema is imposed. Keeping this
intermediate file is what allows the conversion step to be re-run and audited
later.


In [5]:
RAW_H5.parent.mkdir(parents=True, exist_ok=True)
if RAW_H5.exists() and not OVERWRITE:
    raise FileExistsError(f'{RAW_H5} exists; set OVERWRITE=True to replace it')
with h5py.File(RAW_H5, 'w') as f:
    f.attrs.update({
        'env_id': ENV_ID, 'obs_mode': OBS_MODE,
        'control_mode': CONTROL_MODE, 'fps': FPS,
        'data_quality': 'random_smoke_test_not_expert'
    })
    for i, ep in enumerate(episodes):
        group = f.create_group(f'episode_{i:06d}')
        for key, value in ep.items():
            group.create_dataset(key, data=value, compression='gzip')
print('Raw HDF5 written:', RAW_H5)

Raw HDF5 written: /home/bowenyuan/Projects/embodied-ai-learning/datasets/raw/pickcube_smoke.h5


## 2.5 — Convert the collected episodes into LeRobot format

The episodes collected in Python are written to a raw HDF5 file first, then
converted into a LeRobot dataset with an explicit feature schema.

Two things decide whether the result is usable:

1. `features` declares each field's dtype, shape, and names — this is the dataset
   **schema**. It is not the robot **semantics** of those fields: a shape never
   tells you the coordinate frame, the controller mode, or the frequency.
2. `data_quality` is recorded as `random_smoke_test_not_expert`. Random actions
   produce a valid structure with meaningless behaviour, so this dataset is a
   pipeline fixture, not a demonstration set.


In [6]:
if OUTPUT_ROOT.exists():
    if not OVERWRITE:
        raise FileExistsError(f'{OUTPUT_ROOT} exists; set OVERWRITE=True to replace it')
    shutil.rmtree(OUTPUT_ROOT)

features = {
    'observation.state': {
        'dtype': 'float32', 'shape': (obs_dim,),
        'names': [f'state_{i}' for i in range(obs_dim)]
    },
    'action': {
        'dtype': 'float32', 'shape': (action_dim,),
        'names': [f'action_{i}' for i in range(action_dim)]
    },
    'reward': {'dtype': 'float32', 'shape': (1,), 'names': ['reward']},
    'terminated': {'dtype': 'bool', 'shape': (1,), 'names': ['terminated']},
    'truncated': {'dtype': 'bool', 'shape': (1,), 'names': ['truncated']},
    'success': {'dtype': 'bool', 'shape': (1,), 'names': ['success']},
}
dataset = LeRobotDataset.create(
    repo_id=REPO_ID, root=OUTPUT_ROOT, fps=FPS,
    robot_type='maniskill_panda', features=features, use_videos=False
)
for ep in episodes:
    for t in range(len(ep['actions'])):
        dataset.add_frame({
            'observation.state': ep['observations'][t],
            'action': ep['actions'][t],
            'reward': np.asarray([ep['rewards'][t]], dtype=np.float32),
            'terminated': np.asarray([ep['terminated'][t]], dtype=bool),
            'truncated': np.asarray([ep['truncated'][t]], dtype=bool),
            'success': np.asarray([ep['success'][t]], dtype=bool),
            'task': TASK,
        })
    dataset.save_episode()
dataset.finalize()
print('LeRobot dataset written:', OUTPUT_ROOT)

Map:   0%|          | 0/50 [00:00<?, ? examples/s]

Map:   0%|          | 0/50 [00:00<?, ? examples/s]

Map:   0%|          | 0/50 [00:00<?, ? examples/s]

Map:   0%|          | 0/50 [00:00<?, ? examples/s]

Map:   0%|          | 0/50 [00:00<?, ? examples/s]

...

Map:   0%|          | 0/50 [00:00<?, ? examples/s]

Map:   0%|          | 0/50 [00:00<?, ? examples/s]

Map:   0%|          | 0/50 [00:00<?, ? examples/s]

Map:   0%|          | 0/50 [00:00<?, ? examples/s]

LeRobot dataset written: /home/bowenyuan/Projects/embodied-ai-learning/datasets/lerobot/maniskill_pickcube_smoke


In [7]:
loaded = LeRobotDataset(REPO_ID, root=OUTPUT_ROOT)
expected_frames = sum(len(ep['actions']) for ep in episodes)
assert len(loaded) == expected_frames, (len(loaded), expected_frames)
sample = loaded[0]
assert tuple(sample['observation.state'].shape) == (obs_dim,)
assert tuple(sample['action'].shape) == (action_dim,)
print('Validation: PASS')
print('Episodes:', loaded.num_episodes)
print('Frames:', len(loaded))
print('Sample keys:', sorted(sample.keys()))

Validation: PASS
Episodes: 10
Frames: 500
Sample keys: ['action', 'episode_index', 'frame_index', 'index', 'observation.state', 'reward', 'success', 'task', 'task_index', 'terminated', 'timestamp', 'truncated']


## Next: expert demonstrations

For imitation learning, replace the random action provider with ManiSkill's official motion-planning demonstrations or SO-101 teleoperation. Keep the same schema and quality gates, and require an explicit success signal. The random dataset above is deliberately unsuitable as expert training data.